# Ridge, Lasso, and Elastic Net – Medical Insurance Cost Dataset

This notebook demonstrates Ridge, Lasso, and Elastic Net regression using the **Medical Insurance Cost Dataset** (`insurance.csv`).

# Link to download the data
https://www.kaggle.com/datasets/mirichoi0218/insurance

**Dataset columns:** age, sex, bmi, children, smoker, region, charges

## 1. Import Libraries

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error


## 2. Load Dataset

In [ ]:
df = pd.read_csv("insurance.csv")
df.head()

## 3. Basic Exploration

In [ ]:
df.info()

## 4. Encode Categorical Variables

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True) #nominal variables
df_encoded.head()

## 5. Split Features and Target

In [ ]:
X = df_encoded.drop("charges", axis=1)
y = df_encoded["charges"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 6. Feature Scaling (MANDATORY for Regularization)

In [ ]:
scaler = StandardScaler() #after train test split, could scaling be done before splitting the data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 7. Linear Regression (Baseline)

In [8]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred_train = lr.predict(X_train_scaled)
y_pred_test = lr.predict(X_test_scaled)

mse_train = mean_squared_error(y_train, y_pred_train)
mse_test = mean_squared_error(y_test, y_pred_test)



print("Train R2:", r2_score(y_train, y_pred_train))
print("Test R2:", r2_score(y_test, y_pred_test))
print("Train MSE:", mse_train)
print("Test MSE:", mse_test)



print("Coefficients:", lr.coef_)

Train R2: 0.7417255854683333
Test R2: 0.7835929767120722
Train MSE: 37277681.70201867
Test MSE: 33596915.85136148
Coefficients: [ 3.61497541e+03  2.03622812e+03  5.16890247e+02 -9.29310107e+00
  9.55848141e+03 -1.58140981e+02 -2.90157047e+02 -3.49110678e+02]


## 8. Ridge Regression (L2)

In [14]:
ridge = Ridge(alpha=100) #What is alpha -> strength of the penalty -> sometimes also called lambda
ridge.fit(X_train_scaled, y_train)

print("Train R2:", ridge.score(X_train_scaled, y_train))
print("Test R2:", ridge.score(X_test_scaled, y_test))
print("Coefficients:", ridge.coef_)

Train R2: 0.7360496301541095
Test R2: 0.7734197118072562
Coefficients: [3285.53392898 1868.24870437  502.22581052   47.22331739 8718.70365777
 -123.90217648 -161.92184483 -285.86580615]


## 9. Lasso Regression (L1)

In [13]:
lasso = Lasso(alpha=100)
lasso.fit(X_train_scaled, y_train)

print("Train R2:", lasso.score(X_train_scaled, y_train))
print("Test R2:", lasso.score(X_test_scaled, y_test))
print("Coefficients:", lasso.coef_)

Train R2: 0.7410673042245628
Test R2: 0.7806320248423892
Coefficients: [3528.80246822 1892.78999428  424.97875255    0.         9453.06803015
   -0.          -15.58979598 -104.37976391]


## Assignment -> Create another model with ElasticNet and compare the results

## 10. Coefficient Comparison

In [15]:
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Linear": lr.coef_,
    "Ridge": ridge.coef_,
    "Lasso": lasso.coef_
})

coef_df

,Feature,Linear,Ridge,Lasso
0,age,3614.975415,3285.533929,3528.802468
1,bmi,2036.228123,1868.248704,1892.789994
2,children,516.890247,502.225811,424.978753
3,sex_male,-9.293101,47.223317,0.000000
4,smoker_yes,9558.481409,8718.703658,9453.068030
5,region_northwest,-158.140981,-123.902176,-0.000000
6,region_southeast,-290.157047,-161.921845,-15.589796
7,region_southwest,-349.110678,-285.865806,-104.379764


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ["Linear", "Ridge", "Lasso", "ElasticNet"]
features = coef_df["Feature"]

x = np.arange(len(features))
width = 0.2

plt.figure(figsize=(14,6))

plt.bar(x - 1.5*width, coef_df["Linear"], width, label="Linear")
plt.bar(x - 0.5*width, coef_df["Ridge"], width, label="Ridge")
plt.bar(x + 0.5*width, coef_df["Lasso"], width, label="Lasso")
plt.bar(x + 1.5*width, coef_df["ElasticNet"], width, label="ElasticNet")

plt.xticks(x, features, rotation=45, ha="right")
plt.ylabel("Coefficient Value")
plt.title("Coefficient Comparison Across Models")
plt.legend()
plt.tight_layout()
plt.show()


## 11. Effect of Alpha (λ) on Coefficients

In [ ]:

alphas = np.logspace(1, 4, 50)

lasso_coefs = []
ridge_coefs = []

for a in alphas:
    ridge = Ridge(alpha=a)
    ridge.fit(X_train_scaled, y_train)
    ridge_coefs.append(ridge.coef_)

    lasso = Lasso(alpha=a, max_iter=5000)
    lasso.fit(X_train_scaled, y_train)
    lasso_coefs.append(lasso.coef_)

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

plt.figure()
plt.plot(alphas, ridge_coefs)
plt.xscale("log")
plt.xlabel("Alpha")
plt.ylabel("Coefficient Value")
plt.title("Ridge: Coefficients vs Alpha")
plt.show()

plt.figure()
plt.plot(alphas, lasso_coefs)
plt.xscale("log")
plt.xlabel("Alpha")
plt.ylabel("Coefficient Value")
plt.title("Lasso: Coefficients vs Alpha")
plt.show()


## 13. Key Observations

- Ridge shrinks all coefficients but keeps every feature
- Lasso sets some coefficients exactly to zero (feature selection)
- Elastic Net balances stability and sparsity
- Smoking-related features dominate medical cost prediction